In [ ]:
#------------------------------------------------------------------------
#Setup EEG Dataset Loader
#------------------------------------------------------------------------
import os
import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset
import random

class EEGDataset(Dataset):
    def __init__(self, data_dir, crop_length=2000):
        self.data_dir = data_dir
        self.crop_length = crop_length

        # 🔀 Get and shuffle list of CSV files
        self.file_list = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
        random.shuffle(self.file_list)

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file = self.file_list[idx]

        # Extract label from file name prefix (e.g., 0_00001.csv → label 0)
        label = int(file.split('_')[0])

        # ✅ Load only the first 19 EEG channels
        data = pd.read_csv(os.path.join(self.data_dir, file)).iloc[:, :10].values.T  # shape: [channels, time]

        # 🔄 Crop or pad to fixed length
        if data.shape[1] > self.crop_length:
            start = np.random.randint(0, data.shape[1] - self.crop_length)
            data = data[:, start:start + self.crop_length]
        else:
            pad = self.crop_length - data.shape[1]
            data = np.pad(data, ((0, 0), (0, pad)), mode='constant')

        # 🔬 Normalize (z-score)
        data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)

        return torch.tensor(data, dtype=torch.float32), label



In [ ]:
#------------------------------------------------------------------------
#CEEDNet 1D ResNet-18 Backbone
#------------------------------------------------------------------------

import torch.nn as nn

class BasicBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_ch)

        self.downsample = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)

class CEEDNet1DResNet18(nn.Module):
    def __init__(self, in_channels=10, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 64, blocks=2)
        self.layer2 = self._make_layer(64, 128, blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=2, stride=2)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def _make_layer(self, in_ch, out_ch, blocks, stride=1):
        layers = [BasicBlock1D(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_avg_pool(x).squeeze(-1)
        return self.fc(x)


#------------------------------------------------------------------------
#CEEDNet 1D ResNet-18 Backbone
#------------------------------------------------------------------------


In [ ]:
#------------------------------------------------------------------------
#Train the Model
#------------------------------------------------------------------------

from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Dataset split
dataset = EEGDataset(data_dir='/kaggle/input/reduced-csv/signal_csv_reduced', crop_length=2000)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])


# Model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CEEDNet1DResNet18().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

# Trackers
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(1, 51):
    # Training
    model.train()
    total_loss = 0
    correct, total = 0, 0

        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_acc = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Validation
    model.eval()
    val_loss = 0
    correct, total = 0, 0
    with torch.no_grad():
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item()

            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    val_acc = correct / total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    # Print progress
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")


#------------------------------------------------------------------------
#Train the Model
#------------------------------------------------------------------------

from collections import Counter
from torch.utils.data import WeightedRandomSampler

# Count class distribution in training set
targets = [train_ds[i][1] for i in range(len(train_ds))]
class_counts = Counter(targets)

# Create weights for each class (inverse of frequency)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[train_ds[i][1]] for i in range(len(train_ds))]

# Use WeightedRandomSampler to oversample minority class
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# Use the sampler in the DataLoader
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=32)


In [ ]:
#------------------------------------------------------------------------
#Evaluate the Model
#------------------------------------------------------------------------

from sklearn.metrics import classification_report
v
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        preds = torch.argmax(model(x), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print(classification_report(all_labels, all_preds, digits=4))


#------------------------------------------------------------------------
#Evaluate the Model
#------------------------------------------------------------------------


In [ ]:

#======================== plot the raining progress=========================
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Over Time')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Acc')
plt.plot(val_accuracies, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Over Time')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

#======================== final classification report ==========================
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, digits=4, target_names=["Normal", "MCI", "AD"]))


# --------------------------------------
# Confusion Matrix
# --------------------------------------
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# Replace these with your actual predictions and labels
# y_true = list of ground truth labels
# y_pred = list of predicted labels

labels = ['Normal', 'MCI', 'AD']
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

plt.figure(figsize=(6, 5))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.show()
---------------------------------------------------------------------------
